# RunPod Study200: independent experiment cells
A100 SXM 80 GB; both datasets use 200 epochs. Twelve priority runs, six deferred VOC
scheduling repetitions, then six fixed-50 runs (24 full runs / 4,800 epochs).
The cell order is recommended: select any experiment independently after common setup.
No experiment cell invokes another. Training and smoke execution are disabled by default.
Adaptive reviews persist on the volume and block every subsequent launcher, including Run All.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
WORKSPACE = Path('/workspace')
REPO = WORKSPACE / 'kmip'
STUDY_ROOT = WORKSPACE / 'kmip_study200'
DATA_ROOT = WORKSPACE / 'kmip_datasets'
APPROVED_COMMIT = ''  # Fill with the reviewed commit containing this study implementation.
RUN_TRAINING = False
RUN_SMOKE = False
SMOKE_PREDICTION_CHECK = 'warn'  # Temporary smoke-only exception; use 'strict' for the original assertion.
SETUP_ENVIRONMENT = False
PREPARE_DATASETS = False
for path in (STUDY_ROOT, DATA_ROOT):
    path.mkdir(parents=True, exist_ok=True)


## Repository and environment
Use a persistent RunPod volume and a CUDA-enabled PyTorch image. Record its exact image
identifier externally. Select an approved commit, not a moving branch. Existing dirty
checkouts are never reset or deleted. The original notebook's source patch is not used.


In [ ]:
assert APPROVED_COMMIT, 'Set the reviewed immutable repository commit first.'
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/ZoyaAnwar/adaptive-k-MIP-GraphGPS.git',str(REPO)],check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip(), 'Review dirty checkout first.'
subprocess.run(['git','fetch','origin'],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',APPROVED_COMMIT],cwd=REPO,check=True)
APPROVED_COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
os.chdir(REPO)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
assert (REPO/'graphgps/memory_profiling.py').exists()


In [ ]:
import importlib.util
if SETUP_ENVIRONMENT:
    subprocess.run([sys.executable,'-m','pip','install','-r','study/requirements.txt'],check=True)
    import torch
    if importlib.util.find_spec('torch_scatter') is None:
        major_minor='.'.join(torch.__version__.split('+')[0].split('.')[:2])
        tag='cu'+torch.version.cuda.replace('.','')
        wheels=f'https://data.pyg.org/whl/torch-{major_minor}.0+{tag}.html'
        subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:','torch-scatter','-f',wheels],check=True)
    raise RuntimeError('Installation finished: restart the kernel, then leave SETUP_ENVIRONMENT=False before continuing.')
else:
    print('Reusing the environment; dependency validation follows.')


In [ ]:
import torch, torch_geometric, torch_scatter, pykeops
from pykeops.torch import LazyTensor
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
capacity=torch.cuda.get_device_properties(0).total_memory/1024**2 if name else 0
assert 'A100' in name and 'SXM' in name and capacity>75000, (name,capacity)
q=torch.randn(6,4,device='cuda'); keys=torch.randn(9,4,device='cuda')
scores=(LazyTensor(q[:,None,:])*LazyTensor(keys[None,:,:])).sum(-1)
actual=(-scores).argKmin(3,dim=1)
expected=(q@keys.T).topk(3,dim=1).indices
assert torch.equal(actual.sort(-1).values,expected.sort(-1).values)
del q,keys,scores,actual,expected
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
lock=STUDY_ROOT/'environment.freeze.txt'
if lock.exists(): assert lock.read_text()==freeze, 'Environment changed; investigate before resuming.'
else: lock.write_text(freeze)
print(name,capacity,'MiB; CUDA',torch.version.cuda,'PyG',torch_geometric.__version__,'KeOps',pykeops.__version__)


## Prepare both datasets without training
Dataset caching is shared, but experiments/checkpoints remain separate. This cell invokes
only data loaders. Verify splits and fingerprints; no model forward/backward or training
is launched. The repository's PascalVOC mirror and RDKit Peptides loader are reused.


In [ ]:
link=REPO/'datasets'
if link.is_symlink(): assert link.resolve()==DATA_ROOT.resolve()
elif not link.exists(): link.symlink_to(DATA_ROOT,target_is_directory=True)
else: print('Existing datasets directory retained:',link.resolve())
if PREPARE_DATASETS:
    loader_script = """import sys,graphgps
from torch_geometric.graphgym.config import cfg,set_cfg
from graphgps.utils import my_create_loader
from graphgps.adaptive_k.run_identity import dataset_fingerprint
set_cfg(cfg)
cfg.merge_from_file(sys.argv[1])
loaders=my_create_loader()
print('split sizes',[len(loader.dataset) for loader in loaders])
print('training identity',dataset_fingerprint(loaders[0].dataset))
"""
    for dataset in ('PascalVOC-SP','Peptides-func'):
        subprocess.run([sys.executable,'-c',loader_script,f'configs/Study200/{dataset}/linear.yaml'],check=True,cwd=REPO)


## Plan, inspect and report
Initialization creates the ordered manifest only. The resolved run directory is recorded
explicitly because the training code adds config/seed and adaptive namespace components.
Reusing this study root retains run status, review approvals, timings and checkpoints.


In [ ]:
from study.experiment import initialize,next_action,run_job,run_group,group_ids,approve,migrate_manifest
from study.report import generate
MANIFEST=initialize(STUDY_ROOT,APPROVED_COMMIT)
migrate_manifest(MANIFEST)  # v1 backup retained; paths/configs/status/approvals preserved.
manifest=json.loads(MANIFEST.read_text())
import pandas as pd
from IPython.display import display,Image
display(pd.DataFrame([{k:r[k] for k in ('id','dataset','method','seed','epochs','status')} for r in manifest['runs']]))
print(next_action(MANIFEST)['action'])

def show_latest():
    latest=STUDY_ROOT/'reports/latest'
    if not (latest/'summary.json').exists(): return
    report=json.loads((latest/'summary.json').read_text())
    display(pd.DataFrame(report['runs'])[['dataset','method','seed','status','test_score','training_seconds','peak_allocated_mib','peak_reserved_mib','k_average','k_final','calibration_seconds','review_pending','issues','warnings']])
    display(pd.DataFrame(report['aggregates']))
    if report.get('supplementary_profiling'): display(pd.DataFrame(report['supplementary_profiling']))
    if report.get('historical_runs'): display(pd.DataFrame(report['historical_runs']))
    if report.get('historical_alerts'): display(pd.DataFrame(report['historical_alerts']))
    for dataset in ('PascalVOC-SP','Peptides-func'):
        for name in ('learning_train','learning_validation','attention_budget','memory_allocated','memory_reserved','calibration_k','calibration_empirical_coverage','calibration_coverage_lower_bound','calibration_required_k_distribution','calibration_calibration_seconds','calibration_dkw_epsilon','calibration_mass_mean','calibration_mass_p05','calibration_mean_profile_js_divergence','calibration_required_budget_cdf_distance','calibration_required_budget_wasserstein'):
            path=latest/dataset/'figures'/f'{name}.png'
            if path.exists(): display(Image(filename=str(path)))


def execute_cell(number):
    # Batch cells 13/14 report after EACH run, never just at the end.
    result=run_group(MANIFEST,group_ids(number),execute=RUN_TRAINING,
                     retry=False,after_each=show_latest)
    print(json.dumps(result,indent=2,default=str))
    return result


## Optional historical imports (predictive performance only)
Upload the original config/log directories or tar archive to the persistent volume.
Provide the directory containing config.yaml and seed subdirectories, or archive + prefix.
Only full finite 200-epoch histories are accepted. Sources are copied/checksummed; no overwrite.
Historical fixed15 stays separate from fixed50. Historical hardware/runtime is unverified;
no historical timing or missing memory is substituted into A100 comparisons. Seed 0 remains
in descriptive legacy tables; paired study comparisons use seeds 1-3.


In [ ]:
from study.historical import import_logs
HISTORICAL_IMPORTS = []  # Explicit opt-in. Examples below are not executed.
# {'source':'/workspace/legacy/fixed/GPS+kmip-fixed','import_id':'voc_fixed15','method':'fixed15'}
# {'source':'/workspace/legacy/linear/GPS+kmip-linear-50-to-10-linear','import_id':'voc_linear','method':'linear'}
# {'source':'/workspace/legacy/kmip-exponential-results.tar.gz','import_id':'voc_exp','method':'exponential',
#  'prefix':'results/local/PascalVOC-SP/full/exponential/GPS+kmip-exponential-50-to-10-exponential'}
for item in HISTORICAL_IMPORTS:
    import_logs(MANIFEST,**item)


## A100 CUDA/KeOps smoke test before full training
Enable RUN_SMOKE explicitly after datasets/dependencies are ready. This runs two optimizer
steps per method/dataset in separate subprocesses, real validation/test forward passes,
CUDA allocator profiling, CPU checkpoint audit, training resume and standalone state restore.
Each adaptive case performs one m=8500 calibration with the 200-epoch DKW horizon,
on a small deterministic training subset. See smoke/<timestamp>/launcher.log and result.json.
Compilation can dominate pilot time. These smoke-only subset measurements are excluded from
study tables and cannot replace full-run peaks/runtime. A full-batch/full-dataset pilot on
RunPod is still recommended before the first 200-epoch run.

Temporary prediction warning mode leaves numerical consistency **UNVERIFIED**.
All other checks remain mandatory; use SMOKE_PREDICTION_CHECK='strict' to restore the original final assertion.
Comparison diagnostics are saved in each case's prediction_comparison/ directory and result.json.


In [ ]:
from study.smoke import workflow
print(json.dumps(workflow(MANIFEST,execute=RUN_SMOKE,graphs=8,batch_size=2,prediction_check=SMOKE_PREDICTION_CHECK),indent=2))


## Experiment cell 1: PascalVOC-SP / adaptive_mass / seed 1
Full 200-epoch run. Independent selection; validated completed runs are skipped.
After completion, inspect diagnostics and use the manual review utility before any other launch.


In [ ]:
execute_cell(1)


## Experiment cell 2: Peptides-func / linear / seed 1
Full 200-epoch run. Independent selection; validated completed runs are skipped.


In [ ]:
execute_cell(2)


## Experiment cell 3: Peptides-func / exponential / seed 1
Full 200-epoch run. Independent selection; validated completed runs are skipped.


In [ ]:
execute_cell(3)


## Experiment cell 4: Peptides-func / adaptive_mass / seed 1
Full 200-epoch run. Independent selection; validated completed runs are skipped.
After completion, inspect diagnostics and use the manual review utility before any other launch.


In [ ]:
execute_cell(4)


## Experiment cell 5: PascalVOC-SP / adaptive_mass / seed 2
Full 200-epoch run. Independent selection; validated completed runs are skipped.
After completion, inspect diagnostics and use the manual review utility before any other launch.


In [ ]:
execute_cell(5)


## Experiment cell 6: Peptides-func / linear / seed 2
Full 200-epoch run. Independent selection; validated completed runs are skipped.


In [ ]:
execute_cell(6)


## Experiment cell 7: Peptides-func / exponential / seed 2
Full 200-epoch run. Independent selection; validated completed runs are skipped.


In [ ]:
execute_cell(7)


## Experiment cell 8: Peptides-func / adaptive_mass / seed 2
Full 200-epoch run. Independent selection; validated completed runs are skipped.
After completion, inspect diagnostics and use the manual review utility before any other launch.


In [ ]:
execute_cell(8)


## Experiment cell 9: PascalVOC-SP / adaptive_mass / seed 3
Full 200-epoch run. Independent selection; validated completed runs are skipped.
After completion, inspect diagnostics and use the manual review utility before any other launch.


In [ ]:
execute_cell(9)


## Experiment cell 10: Peptides-func / linear / seed 3
Full 200-epoch run. Independent selection; validated completed runs are skipped.


In [ ]:
execute_cell(10)


## Experiment cell 11: Peptides-func / exponential / seed 3
Full 200-epoch run. Independent selection; validated completed runs are skipped.


In [ ]:
execute_cell(11)


## Experiment cell 12: Peptides-func / adaptive_mass / seed 3
Full 200-epoch run. Independent selection; validated completed runs are skipped.
After completion, inspect diagnostics and use the manual review utility before any other launch.


In [ ]:
execute_cell(12)


## Experiment cell 13: Deferred PascalVOC scheduling repetitions
Linear and exponential, seeds 1-3 (six full 200-epoch runs).
Runs only this group; updates/displays reports after every run and stops on failure or review gate.


In [ ]:
execute_cell(13)


## Experiment cell 14: Final fixed k=50 experiments
Both datasets, seeds 1-3 (six full 200-epoch runs).
Runs only this group; updates/displays reports after every run and stops on failure or review gate.


In [ ]:
execute_cell(14)


## Persistent manual adaptive review (utility; no training)
Run only after reviewing the selected-k trajectory, required-k histograms, retained-mass
coverage, DKW parameters, observed counts, overhead, memory and learning curves. Run All
never approves a gate: APPROVE_REVIEW defaults False each time this cell executes.
An approval is tied to artifact fingerprints; changed results require a fresh review.


In [ ]:
APPROVE_REVIEW = False  # Change and execute this cell deliberately after inspecting results.
pending=next_action(MANIFEST)
if pending['action']=='awaiting_review' and APPROVE_REVIEW:
    run_id=pending['run_id']
    confirmation=input(f'Type APPROVE {run_id}: ')
    approve(MANIFEST,run_id,confirmation)
    generate(MANIFEST)
else:
    print(json.dumps(pending,indent=2,default=str))


## Optional supplementary profiling (never a replacement)
Disabled by default. Uses each dataset's configured full batch size, two optimizer warmup
steps and two measured train/evaluation windows (baseline k endpoints, or adaptive initial
and calibrated k). It uses an explicit small subset and freshly initialized models, not
trained checkpoints or full-dataset trajectories. Calibration samples still number 8500,
with the 200-epoch DKW horizon. The resulting peaks are subset-window measurements only;
they cannot establish full-run peaks or runtime, or replace cells 13/14. Runtime is not
extrapolated to 200 epochs. Run all 24 full jobs for matched final efficiency comparisons.


In [ ]:
RUN_SUPPLEMENTARY_PROFILING = False
if RUN_SUPPLEMENTARY_PROFILING:
    print(workflow(MANIFEST,execute=True,graphs=400,batch_size=None,profile=True,warmup_steps=2))
    generate(MANIFEST)
    show_latest()


## Recovery and independent report regeneration
Use run_job(MANIFEST, '<exact ID>', execute=RUN_TRAINING, retry=True) only after investigating
a failed/interrupted attempt. Never remove a launch lock until verifying no process remains.
A failed/partial adaptive run blocks other cells until that run is recovered and reviewed.
Checkpoint CPU-load audits and hashes are required for completed study runs; for pre-audit
local study outputs use python -m study.checkpoints <run_dir> --best-epoch <epoch> only on
trusted local checkpoints after checking the earliest best validation epoch.

Reports can always be regenerated without training: python -m study.report <manifest.json>.
Short optional supplementary profiling must use separate profiling_only outputs; it cannot
satisfy any of the 24 full-run jobs. The deferred cells remain 200 epochs by default.


In [ ]:
summary=generate(MANIFEST)
show_latest()
print('Persistent reports:',STUDY_ROOT/'reports/latest')


## Final or interim regeneration and backup
The same report generator is independent of the notebook and training:
`python -m study.report /workspace/kmip_study200/manifest.json`.
It exports CSV/LaTeX tables and PDF/300-dpi PNG figures under each dataset and
cross_dataset summaries. Snapshots preserve interim reports. Missing, failed or partial
runs are listed rather than silently counted. F1 and AP are not pooled across datasets.

Back up the entire study root, including approvals, session logs, best and latest
checkpoints, memory JSON, calibration NPZ/JSON and environment lock. Verify the backup
before manually terminating the RunPod instance. This notebook never controls billing.
